

> Add blockquote



In [3]:
!pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.0/935.0 kB 10.3 MB/s  0:00:00


#  1. Đọc dữ liệu vào chương trình

In [4]:
print("Nguyễn Duy Bình - 6451071005")
import pandas as pd
import time

start = time.time()

df = pd.read_excel(
    "Online Retail.xlsx",
    engine="calamine"
)

print("Đọc dữ liệu thành công!")
print("Kích thước:", df.shape)
print("Thời gian đọc:", round(time.time() - start, 2), "giây")

display(df.head())

Nguyễn Duy Bình - 6451071005
Đọc dữ liệu thành công!
Kích thước: (541909, 8)
Thời gian đọc: 13.84 giây


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# 2. Tiền xử lý dữ liệu: cắt bỏ các ký tự thừa ở tên mặt hàng mua (cột Description), xóa các dòng dữ liệu không có số hóa đơn (cột InvoiceNo) và chuyển nó về kiểu dữ liệu chuỗi



In [5]:
# 1. Loại bỏ khoảng trắng thừa ở đầu và cuối tên mặt hàng
df["Description"] = df["Description"].str.strip()

# 2. Xóa các dòng không có số hóa đơn
df = df.dropna(subset=["InvoiceNo"]).copy()

# 3. Chuyển InvoiceNo sang kiểu chuỗi
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

# Kiểm tra kết quả
print(df[["InvoiceNo", "Description"]].head())
print("Số dòng sau tiền xử lý:", len(df))
print("Kiểu dữ liệu InvoiceNo:", df["InvoiceNo"].dtype)

  InvoiceNo                          Description
0    536365   WHITE HANGING HEART T-LIGHT HOLDER
1    536365                  WHITE METAL LANTERN
2    536365       CREAM CUPID HEARTS COAT HANGER
3    536365  KNITTED UNION FLAG HOT WATER BOTTLE
4    536365       RED WOOLLY HOTTIE WHITE HEART.
Số dòng sau tiền xử lý: 541909
Kiểu dữ liệu InvoiceNo: object


# 3. Trong dữ liệu đã cho, có một số hóa đơn là hóa đơn tín dụng thay vì là hóa đơn ghi nợ vì vậy hãy xóa những hóa đơn đó. Chúng được xác định với ký tự ‘C’ chứa trong số hóa đơn InvoiceNo.


In [6]:
# Đếm số hóa đơn tín dụng trước khi xóa
credit_invoices = df[
    df["InvoiceNo"].str.contains("C", na=False)
]

print("Số dòng thuộc hóa đơn tín dụng:", len(credit_invoices))

# Xóa các dòng thuộc hóa đơn có chứa ký tự C
df = df[~df["InvoiceNo"].str.contains("C", na=False)].copy()

print("Số dòng sau khi xóa hóa đơn tín dụng:", len(df))

# Kiểm tra lại
print("Số hóa đơn còn chứa C:",
      df["InvoiceNo"].str.contains("C", na=False).sum())

Số dòng thuộc hóa đơn tín dụng: 9288
Số dòng sau khi xóa hóa đơn tín dụng: 532621
Số hóa đơn còn chứa C: 0


# 4. Thống kê số dòng dữ liệu theo từng quốc gia.


In [7]:
# Thống kê số dòng theo quốc gia
country_counts = df["Country"].value_counts()

print(country_counts)

Country
United Kingdom          487622
Germany                   9042
France                    8408
EIRE                      7894
Spain                     2485
Netherlands               2363
Belgium                   2031
Switzerland               1967
Portugal                  1501
Australia                 1185
Norway                    1072
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Unspecified                446
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     295
Hong Kong                  284
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         58


# 5. Lấy ra dữ liệu hóa đơn từ nước Anh ‘United Kingdom’ và gom nhóm cột Số lượng mua (Quantity) theo Số hóa đơn (InvoiceNo) và Tên mặt hàng (Description).


In [8]:
# Lọc dữ liệu của Anh
uk_data = df[df["Country"] == "United Kingdom"].copy()

print("Số dòng dữ liệu tại Anh:", len(uk_data))

# Gom nhóm theo số hóa đơn và tên mặt hàng
uk_grouped = (
    uk_data.groupby(["InvoiceNo", "Description"], as_index=False)
           ["Quantity"].sum()
)

# Hiển thị kết quả
print(uk_grouped.head(20))

# Kiểm tra kích thước
print("Kích thước dữ liệu sau gom nhóm:", uk_grouped.shape)

Số dòng dữ liệu tại Anh: 487622
   InvoiceNo                          Description  Quantity
0     536365       CREAM CUPID HEARTS COAT HANGER         8
1     536365    GLASS STAR FROSTED T-LIGHT HOLDER         6
2     536365  KNITTED UNION FLAG HOT WATER BOTTLE         6
3     536365       RED WOOLLY HOTTIE WHITE HEART.         6
4     536365         SET 7 BABUSHKA NESTING BOXES         2
5     536365   WHITE HANGING HEART T-LIGHT HOLDER         6
6     536365                  WHITE METAL LANTERN         6
7     536366            HAND WARMER RED POLKA DOT         6
8     536366               HAND WARMER UNION JACK         6
9     536367        ASSORTED COLOUR BIRD ORNAMENT        32
10    536367   BOX OF 6 ASSORTED COLOUR TEASPOONS         6
11    536367       BOX OF VINTAGE ALPHABET BLOCKS         2
12    536367         BOX OF VINTAGE JIGSAW BLOCKS         3
13    536367                  DOORMAT NEW ENGLAND         4
14    536367    FELTCRAFT PRINCESS CHARLOTTE DOLL         8
15    53

# 6. Chuyển đổi dữ liệu về dạng hot encoding, với mỗi dòng dữ liệu là một hóa đơn.

In [9]:
# Tạo bảng: mỗi hóa đơn là một dòng, mỗi mặt hàng là một cột
basket_quantity = uk_grouped.pivot(
    index="InvoiceNo",
    columns="Description",
    values="Quantity"
).fillna(0)

# Chuyển thành Hot Encoding
# Chỉ đánh dấu 1 khi số lượng mua lớn hơn 0
hot_encoded = (basket_quantity > 0).astype(int)

# Hiển thị kết quả
print(hot_encoded.head())

# Kiểm tra kích thước
print("Số hóa đơn:", hot_encoded.shape[0])
print("Số mặt hàng:", hot_encoded.shape[1])

# Lưu kết quả ra file CSV
hot_encoded.to_csv("Online_Retail_Hot_Encoding.csv", encoding="utf-8-sig")

print("Đã lưu file Online_Retail_Hot_Encoding.csv")

Description  *Boombox Ipod Classic  *USB Office Mirror Ball  \
InvoiceNo                                                     
536365                           0                        0   
536366                           0                        0   
536367                           0                        0   
536368                           0                        0   
536369                           0                        0   

Description  10 COLOUR SPACEBOY PEN  12 COLOURED PARTY BALLOONS  \
InvoiceNo                                                         
536365                            0                           0   
536366                            0                           0   
536367                            0                           0   
536368                            0                           0   
536369                            0                           0   

Description  12 DAISY PEGS IN WOOD BOX  12 EGG HOUSE PAINTED WOOD  \
InvoiceNo          